# ReefScan AI — All-in-One Colab Pipeline (GPU)

Runs every phase end-to-end on a **free Colab T4**: dataset download → enhancement benchmark → fish + coral training (4 backbones) → segmentation → XAI → analytics → PDF reports.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# --- (Colab only) clone the repository ---------------------------------------
# Clone the repo (uncomment and run ONCE if /content/reefscan does not exist):
# !git clone https://github.com/Jayant1711/reefscan.git /content/reefscan
import os
if os.path.isdir("/content/reefscan"):
    %cd /content/reefscan
else:
    print("NOTE: /content/reefscan missing — uncomment the git clone line above, run it, then re-run this cell.")
!pip install -q gdown pyarrow reportlab streamlit


In [ ]:
# --- Environment check -------------------------------------------------------
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
PROJECT = pathlib.Path("/content/reefscan") if IN_COLAB else pathlib.Path.cwd()
print("Colab:", IN_COLAB, "| project:", PROJECT)
if IN_COLAB:
    import os
    from google.colab import drive
    # Mount Drive so datasets/checkpoints survive session resets (optional).
    # drive.mount("/content/drive")


In [ ]:
import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.config import load_config, ensure_output_dirs

cfg = load_config(PROJECT / "configs/default.yaml")
ensure_output_dirs(cfg)
import torch
print("CUDA available:", torch.cuda.is_available())


## 1. Datasets (~13.6 GB total — DeepFish is the big one)

In [ ]:
# Coral (770 MB) and DeepFish (7.1 GB) are automatic.
# FishNet comes from the fishnet.ai S3 mirror (resumable curl, no quota);
# download manually (see docs/DATASETS.md) into datasets/fishnet/raw/.
!python download_data.py --dataset all
!python download_data.py --prepare-fishnet


## 2. Enhancement benchmark (Phase 1)

In [ ]:
!python evaluate.py --enhancement --data datasets/fishnet/curated/train
import json, pathlib
print(json.dumps(json.load(open("results/enhancement/enhancement_report.json")), indent=2))


## 3. Species classification — 4 backbones (Phase 2)

In [ ]:
# 15 epochs x 4 models ≈ 3-5 h on T4. Trim with --set fish.epochs=5 for a quicker pass.
!python train.py --task fish --model all


In [ ]:
import pandas as pd
pd.read_csv("results/fish/model_leaderboard.csv")


## 4. Coral bleaching — 4 backbones + WGBS (Phase 3)

In [ ]:
!python train.py --task coral --model all
pd.read_csv("results/coral/model_leaderboard.csv")


## 5. Fish segmentation for detection boxes (Phase 3.5)

In [ ]:
!python train.py --task seg

## 6. Explainability (Phase 4) + inference demo (Phase 5)

In [ ]:
import glob, cv2
sample = sorted(glob.glob("datasets/coral/imagefolders/test/*/*.jpg"))[0]
!python evaluate.py --xai --image {sample}
!python inference.py --mode image --input {sample} --cam --report


## 7. Launch the dashboard inside Colab

In [ ]:
from pyngrok import ngrok  # or use the link printed by cloudflared below
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501


## 8. Save checkpoints to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r models /content/drive/MyDrive/reefscan_models/
# !cp -r results reports /content/drive/MyDrive/reefscan_outputs/
